In [1]:
import pandas as pd
import pyreadstat

# Load original dataset
df, meta = pyreadstat.read_dta("../data/raw/cdi_household_clean_nopii.dta")

# Define direct-mapped variables
direct_vars = [
    "hhmem_size", "landuseha", "yrs_grow", "lead_vill", "cocoa_produce_kg", "cocoa_produce_main_kg",
    "cocoa_produce_light_kg", "cocoa_sell_kg", "shade_why1", "hhgroup_m18", "hhgroup_f18", "hhgroup_m17",
    "hhgroup_f17", "cocoa_price_kg", "screen_sharecrop",
    "region", "district", "vill", "village", "community_type", "ethnicity","screen_sharecrop", "religion",
    # Household groups 
    "hhgroup_m18", "hhgroup_f18", "hhgroup_m17", "hhgroup_f17",
    # Simple flags (Pages 1-2, adjust if needed)
    "femhead", "econstatus",
    #Cocoa Variety
    "cocoa_variety_1", "cocoa_variety_2", "cocoa_variety_3", "cocoa_variety_4", "cocoa_variety_5", 
    #Farm Rights
    "farm_rights_1", "farm_rights_2", "farm_rights_3", "farm_rights_4", "farm_rights_5", 
    #Farm Tenure
    "farm_tenure_1", "farm_tenure_2", "farm_tenure_3", "farm_tenure_4", "farm_tenure_5",
    #NonCocoa WhyNo
    "noncocoa_whyno_1", "noncocoa_whyno_2", "noncocoa_whyno_3", "noncocoa_whyno_4", "noncocoa_whyno_5",
    #Farm Swamp
    "farm_swamp_1", "farm_swamp_2", "farm_swamp_3", "farm_swamp_4", "farm_swamp_5",


]

# Subset and export
df_direct = df[direct_vars]
df_direct.to_csv("C:/Users/Mohamad/Documents/GitHub/GAFL_RESTORE_MKM/Data/processed/CDI_HOUSEHOLD_CHOSEN.csv", index=False)

print(f"Exported {len(df_direct.columns)} direct-mapped variables to CSV.")


Exported 54 direct-mapped variables to CSV.


In [2]:
import pandas as pd
import pyreadstat
import os

# 1. Load the previously exported CSV of direct-mapped variables
direct_csv_path = "C:/Users/Mohamad/Documents/GitHub/GAFL_RESTORE_MKM/Data/processed/CDI_HOUSEHOLD_CHOSEN.csv"
df_direct = pd.read_csv(direct_csv_path)

# 2. Load original dataset (for engineered features)
df, meta = pyreadstat.read_dta("../data/raw/cdi_household_clean_nopii.dta")

# 3. Define feature mapping (new feature: [source columns])
FEATURE_MAP = {
    # First group (Page 1-2)
    'used_fert?': [f'fert_{i}' for i in [1,2,3,4,5]],
    'used_fert_lqd?': [f'fert_lqd_{i}' for i in [1,2,3,4,5]],
    'herbicide_yn': [f'herbicide_{i}' for i in range(1,6)],
    'pesticide_yn': [f'pesticide_{i}' for i in range(1,6)],
    'fungicide_yn': [f'fungicide_{i}' for i in range(1,6)],
    'weeding_yn': [f'labor_hh_3_{i}' for i in range(1,6)],
    'pruning_yn': [f'labor_hh_5_{i}' for i in range(1,6)],
    
    # Second group (Page 4-5)
    'cocoa_intercrop': [f'cocoa_intercrop_{i}' for i in range(1,6)],
    'farm_rights': [f'farm_rights_{i}' for i in range(1,6)],
    'disease_loss': [f'disease_loss_{i}' for i in range(1,6)],
    'pest_loss': [f'pest_loss_{i}' for i in range(1,6)],
    
    # Replant group (Page 6)
    'replant': [f'replant_{i}' for i in range(1,6)],
    'shadetree_yn': [f'shadetree_yn_{i}' for i in range(1,6)],
    'doc_land': [f'doc_land_{i}' for i in range(1,6)],
    
    # Non-cocoa group (Page 7)
    'noncocoa': [f'noncocoa_{i}' for i in range(1,6)],
    
    # Final group (Page 11)
    'cocoa_yn': [f'cocoa_yn_{i}' for i in range(1,6)],
}

def compute_or_feature(series_list):
    """
    Given a list of Series, combine them to produce an OR-aggregated feature.
    For each row, if any value is > 0, the result is 1; otherwise 0.
    """
    temp_df = pd.concat(series_list, axis=1)
    return temp_df.fillna(0).astype(float).gt(0).any(axis=1).astype(int)

# 4. Compute engineered features into a dictionary
engineered_features = {}

for new_feature, source_cols in FEATURE_MAP.items():
    # Select only valid columns from the original dataset
    valid_cols = [col for col in source_cols if col in df.columns]
    if not valid_cols:
        engineered_features[new_feature] = pd.Series(0, index=df.index)
    else:
        series_list = []
        for col in valid_cols:
            s = df[col]
            # If the column is of object type, convert to numeric
            if s.dtype == 'object':
                s = pd.to_numeric(s, errors='coerce')
            series_list.append(s)
        engineered_features[new_feature] = compute_or_feature(series_list)

# Build a DataFrame from the engineered features
df_engineered = pd.DataFrame(engineered_features, index=df.index)

# 5. Merge the direct mapped variables with the engineered features
df_combined = pd.concat([df_direct, df_engineered], axis=1)

# 6. Export the combined DataFrame to CSV (overwriting the previous file)
output_csv_path = "C:/Users/Mohamad/Documents/GitHub/GAFL_RESTORE_MKM/Data/processed/CDI_HOUSEHOLD_CHOSEN.csv"
os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
df_combined.to_csv(output_csv_path, index=False)

print("Direct mapped and engineered features combined and CSV saved!")


Direct mapped and engineered features combined and CSV saved!


In [3]:
# 1. Read the existing CSV
direct_csv_path = "C:/Users/Mohamad/Documents/GitHub/GAFL_RESTORE_MKM/Data/processed/CDI_HOUSEHOLD_CHOSEN.csv"
df_chosen = pd.read_csv(direct_csv_path)

# 2. Load the original dataset (for farm-area columns)
df, meta = pyreadstat.read_dta("../data/raw/cdi_household_clean_nopii.dta")

# 3. Define the farm-area columns and ensure they exist in df
farm_cols = ["farm_a", "farm_a_1", "farm_a_2", "farm_a_3", "farm_a_4", "farm_a_5"]
valid_farm_cols = [col for col in farm_cols if col in df.columns]

# Convert these columns to numeric (in case they're objects/strings)
for col in valid_farm_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Create a new column in df_chosen indicating if any farm > 1.21
df_chosen["any_farm_above_1_21"] = (df[valid_farm_cols] > 1.21).any(axis=1).astype(int)

# 5. Overwrite the CSV with the updated DataFrame
df_chosen.to_csv(direct_csv_path, index=False)
print("Updated the CSV with the new 'any_farm_above_1_21' column!")

Updated the CSV with the new 'any_farm_above_1_21' column!


In [4]:
import pandas as pd
import pyreadstat

# Paths
direct_csv_path = "C:/Users/Mohamad/Documents/GitHub/GAFL_RESTORE_MKM/Data/processed/CDI_HOUSEHOLD_CHOSEN.csv"
dta_path = "../data/raw/cdi_household_clean_nopii.dta"

# 1. Read the existing CSV (the "chosen" dataset)
df_chosen = pd.read_csv(direct_csv_path)

# 2. Load the original .dta file, which has the repeated columns
df, meta = pyreadstat.read_dta(dta_path)

# Columns for household relationship (which indicates who is the head)
relate_cols = [f"hh_relate_{i}" for i in range(1,16)]

# Make sure these columns exist in df (skip any that might not exist)
valid_relate_cols = [c for c in relate_cols if c in df.columns]

# Convert them to numeric (in case they're strings)
df[valid_relate_cols] = df[valid_relate_cols].apply(pd.to_numeric, errors="coerce")

# 3. Identify which household member is "head" (hh_relate_x == 1)
#    idxmax(axis=1) will return the column name of the *first* maximum value (i.e., first '1').
#    If a row has no 1, idxmax defaults to the first column; we'll fix that by checking .any(axis=1).
df_relate_bool = (df[valid_relate_cols] == 1)
head_col_series = df_relate_bool.idxmax(axis=1)  # e.g., "hh_relate_3"

# If a row has no 1 at all, set the result to None instead of the first column
no_head_mask = ~df_relate_bool.any(axis=1)
head_col_series[no_head_mask] = None

# Extract the numeric part of the column name ("3" from "hh_relate_3")
df["head_index"] = head_col_series.str.extract(r"(\d+)$")
df["head_index"] = pd.to_numeric(df["head_index"], errors="coerce")

# 4. Helper function to pick out the correct column, e.g. hhmem_age_3
def get_head_value(row, prefix):
    """Return the value from prefix + head_index for this row (or NaN if no head)."""
    i = row["head_index"]
    if pd.isna(i):
        return pd.NA
    col_name = f"{prefix}{int(i)}"
    if col_name in df.columns:
        return row[col_name]
    return pd.NA

# For convenience, convert the needed hhmem columns to numeric (or keep as object if needed).
# Age columns
for i in range(1,16):
    col = f"hhmem_age_{i}"
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Education columns
for i in range(1,16):
    col = f"hhmem_edu_level_{i}"
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Birthplace columns (might be string/categorical, so we can leave as is or convert if you prefer)
# for i in range(1,16):
#     col = f"hhmem_born_{i}"
#     if col in df.columns:
#         df[col] = pd.to_numeric(df[col], errors="coerce")  # or leave as string if it's a location

# 5. Create the new columns in df
df["hh_head_age"] = df.apply(lambda r: get_head_value(r, "hhmem_age_"), axis=1)
df["hh_head_education"] = df.apply(lambda r: get_head_value(r, "hhmem_edu_level_"), axis=1)
df["hh_head_birthplace"] = df.apply(lambda r: get_head_value(r, "hhmem_born_"), axis=1)

# 6. Copy these columns over to df_chosen (matching row-by-row)
df_chosen["hh_head_age"] = df["hh_head_age"]
df_chosen["hh_head_education"] = df["hh_head_education"]
df_chosen["hh_head_birthplace"] = df["hh_head_birthplace"]

# Save the updated CSV
df_chosen.to_csv(direct_csv_path, index=False)
print("Added hh_head_age, hh_head_education, and hh_head_birthplace to the chosen dataset!")


Added hh_head_age, hh_head_education, and hh_head_birthplace to the chosen dataset!


In [5]:
csv_path = "C:/Users/Mohamad/Documents/GitHub/GAFL_RESTORE_MKM/Data/processed/CDI_HOUSEHOLD_CHOSEN.csv"

# 1. Read the current dataset
df_chosen = pd.read_csv(csv_path)

# 2. Create the new column (handle non-numeric or zero landuseha if needed)
df_chosen["cocoa_yield_ha"] = (
    pd.to_numeric(df_chosen["cocoa_produce_kg"], errors="coerce") /
    pd.to_numeric(df_chosen["landuseha"], errors="coerce")
)

# 3. Save back to CSV
df_chosen.to_csv(csv_path, index=False)
print("Added 'cocoa_yield_ha' to the dataset!")

Added 'cocoa_yield_ha' to the dataset!


In [10]:
# Paths
csv_path = "C:/Users/Mohamad/Documents/GitHub/GAFL_RESTORE_MKM/Data/processed/CDI_HOUSEHOLD_CHOSEN.csv"
dta_path = "../data/raw/cdi_household_clean_nopii.dta"

# Load the data
df_chosen = pd.read_csv(csv_path)
df_full, meta = pyreadstat.read_dta(dta_path)

# Define bracket columns and tree count columns
AGE_BRACKETS = {
    "cocoa_ages_0_5":  [f"cocoa_ages_0_5_{i}"  for i in range(1, 6)],
    "cocoa_ages_6_10": [f"cocoa_ages_6_10_{i}" for i in range(1, 6)],
    "cocoa_ages_11_19":[f"cocoa_ages_11_19_{i}"for i in range(1, 6)],
    "cocoa_ages_20_30":[f"cocoa_ages_20_30_{i}"for i in range(1, 6)],
    "cocoa_ages_31":   [f"cocoa_ages_31_{i}"   for i in range(1, 6)]
}
TREE_COUNT_COLS = [f"cocoa_num_{i}" for i in range(1, 6)]

# Convert tree counts to numeric
for col in TREE_COUNT_COLS:
    if col in df_full.columns:
        df_full[col] = pd.to_numeric(df_full[col], errors="coerce")
    else:
        df_full[col] = pd.Series([0] * len(df_full))

# Create columns for weighted age brackets
for bracket_name, bracket_cols in AGE_BRACKETS.items():
    # Initialize accumulator for total weighted trees in this age bracket
    weighted_tree_sum = pd.Series([0.0] * len(df_full))
    # Initialize accumulator for total number of trees counted
    total_trees = pd.Series([0.0] * len(df_full))
    
    # Process each farm
    for i in range(5):  # Farm 1-5
        # Convert age percentages to numeric
        if bracket_cols[i] in df_full.columns:
            bracket_pct = pd.to_numeric(df_full[bracket_cols[i]], errors="coerce") / 100.0  # Convert to proportion
        else:
            continue  # Skip if column doesn't exist
            
        # Get tree count for this farm
        if TREE_COUNT_COLS[i] in df_full.columns:
            tree_count = pd.to_numeric(df_full[TREE_COUNT_COLS[i]], errors="coerce")
        else:
            continue  # Skip if column doesn't exist
            
        # Calculate weighted value (only where both values exist)
        valid_mask = (~pd.isna(bracket_pct)) & (~pd.isna(tree_count))
        
        # Number of trees in this age bracket = percentage * total trees
        trees_in_bracket = pd.Series([0.0] * len(df_full))
        trees_in_bracket[valid_mask] = bracket_pct[valid_mask] * tree_count[valid_mask]
        
        # Add to accumulators
        weighted_tree_sum += trees_in_bracket
        total_trees[valid_mask] += tree_count[valid_mask]
    
    # Calculate weighted percentage across all farms
    # This represents the percentage of all trees that fall into this age bracket
    df_chosen[f"{bracket_name}_weighted"] = (weighted_tree_sum / total_trees * 100).fillna(0)

# Add a validation column that sums all weighted percentages
weighted_cols = [f"{bracket_name}_weighted" for bracket_name in AGE_BRACKETS]
df_chosen["age_weighted_total"] = df_chosen[weighted_cols].sum(axis=1)

# Check if percentages sum to approximately 100%
close_to_100 = ((df_chosen["age_weighted_total"] >= 99) & 
                (df_chosen["age_weighted_total"] <= 101)).sum()
total_non_zero = (df_chosen["age_weighted_total"] > 0).sum()

print(f"Calculated weighted age distribution by number of trees")
print(f"Rows with total close to 100% (±1%): {close_to_100} out of {total_non_zero} non-zero rows")

# Check for any extreme values
if (df_chosen["age_weighted_total"] > 101).any():
    print(f"Warning: {(df_chosen['age_weighted_total'] > 101).sum()} rows have total > 101%")
    print(f"Max total: {df_chosen['age_weighted_total'].max():.2f}%")

# Save the updated dataframe
df_chosen.to_csv(csv_path, index=False)
print("Added weighted cocoa age brackets to your chosen dataset!")

Calculated weighted age distribution by number of trees
Rows with total close to 100% (±1%): 1482 out of 1482 non-zero rows
Added weighted cocoa age brackets to your chosen dataset!


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# 1) Read your chosen dataset
csv_path = "C:/Users/Mohamad/Documents/GitHub/GAFL_RESTORE_MKM/Data/processed/CDI_HOUSEHOLD_CHOSEN.csv"
df = pd.read_csv(csv_path)

# 2) Define columns & strategies
#    Adjust these lists to match actual data usage.

# Columns that are strictly "multi-value" (space-separated) and need multi-label binarization
MULTI_VALUE_COLS = [
    "cocoa_variety_1", "cocoa_variety_2", "cocoa_variety_3", "cocoa_variety_4", "cocoa_variety_5",
    "farm_rights_1", "farm_rights_2", "farm_rights_3", "farm_rights_4", "farm_rights_5",
    "farm_tenure_1", "farm_tenure_2", "farm_tenure_3", "farm_tenure_4", "farm_tenure_5",
    "noncocoa_whyno_1", "noncocoa_whyno_2", "noncocoa_whyno_3", "noncocoa_whyno_4", "noncocoa_whyno_5",
    "farm_swamp_1", "farm_swamp_2", "farm_swamp_3", "farm_swamp_4", "farm_swamp_5"
    # Only include these if they can indeed have multiple values in one cell (e.g. "10 13")
]

# Columns that are single-valued categorical (might use simple one-hot or label encoding).
# Example: region, district, lead_vill, ethnicity, religion, etc.
# If they sometimes have multiple space-separated values, move them to MULTI_VALUE_COLS.
SINGLE_CAT_COLS = [
    "yrs_grow",
    "lead_vill",
    "shade_why1",
    "region",
    "district",
    "ethnicity",
    "religion",
    "hh_head_education",
    "hh_head_birthplace"
]

# Column that is basically just an ID or numeric code you might want to drop or keep as numeric
ID_COLS = [
    "vill"
]

# Column with exactly 2 categories, where you want "RESTORE"=1 else 0
BINARY_COL = "community_type"

# 3) Handle BINARY_COL directly
#    If 'community_type' is strictly "RESTORE" vs something else:
df[BINARY_COL] = df[BINARY_COL].apply(lambda x: 1 if str(x).strip().upper() == "RESTORE" else 0)

# 4) Define a helper to split multi-value cells into lists
def split_multi_values(cell):
    """Split a string by whitespace into a list of tokens; return empty list if null/NaN."""
    if isinstance(cell, str) and cell.strip():
        return cell.split()
    return []

# 5) MultiLabelBinarizer for the MULTI_VALUE_COLS
mlb = MultiLabelBinarizer(sparse_output=False)
new_multi_dfs = []

for col in MULTI_VALUE_COLS:
    if col not in df.columns:
        # If the column doesn't exist, skip it
        continue
    
    # 5a) Convert the column to a list of lists
    lists_of_values = df[col].apply(split_multi_values)
    
    # 5b) Fit/transform with MultiLabelBinarizer
    #     This creates new columns for each unique token in that column
    binarized = mlb.fit_transform(lists_of_values)
    new_cols = [f"{col}_{cls}" for cls in mlb.classes_]
    
    # Create a DataFrame with the new binary columns
    temp_df = pd.DataFrame(binarized, columns=new_cols, index=df.index)
    new_multi_dfs.append(temp_df)

# 5c) Concatenate all newly created dummy data
if new_multi_dfs:
    df_multi_expanded = pd.concat(new_multi_dfs, axis=1)
else:
    df_multi_expanded = pd.DataFrame(index=df.index)

# 6) Drop the original multi-value columns from df
df.drop(columns=MULTI_VALUE_COLS, inplace=True, errors="ignore")

# 7) Combine the expanded multi-value dummies back into df
df = pd.concat([df, df_multi_expanded], axis=1)

# 8) One-hot encoding for single-valued categorical columns
#    If you prefer a numeric label encoding, you can do that instead.
df = pd.get_dummies(df, columns=SINGLE_CAT_COLS, drop_first=False, dtype=int)

# 9) Decide what to do with ID_COLS
#    Typically, numeric IDs are not helpful for ML, so you might drop them if they are unique per row.
df.drop(columns=ID_COLS, inplace=True, errors="ignore")

# 10) Save the updated DataFrame
df.to_csv(csv_path, index=False)
print("Successfully binarized multi-value columns, one-hot encoded single categories, and updated the CSV!")


Successfully binarized multi-value columns, one-hot encoded single categories, and updated the CSV!
